**Today's four questions**

1. What is a *typical* listing?
2. How much do the listings *differ* from each other?
3. Which listings are so unusual that we should look at them separately?
4. Do two columns, like the price and the number of guests, *move together*?

In [2]:
import pandas as pd

listings = pd.read_csv('data/listings_clean.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])

print(listings.shape)
print(listings['price'].dtype, listings['last_review'].dtypes)

(4574, 73)
float64 datetime64[us]


## The mean and the median: why they are different

In [4]:
five = pd.Series([900, 1000, 1100, 1200, 1300])

print(five.mean(), five.median())

1100.0 1100.0


In [6]:
six = pd.Series([900, 1000, 1100, 1200, 1300, 40000])

print(round(six.mean(), 1), six.median())

7583.3 1150.0


- **Median:** the typical value, when a few values are far away from the rest. It is hard to push around.
- **Mean:** when you want to calculate something with *all* the values, like a total, or when the values are spread evenly. Mean times count gives you the sum.

In [7]:
print(listings['price'].mean().round(1))
print(listings['price'].median())

2327.6
1791.0


## Percentiles and the spread

In [8]:
listings['price'].describe().round(1)

count     2957.0
mean      2327.6
std       2160.4
min        120.0
25%       1071.1
50%       1791.0
75%       3003.8
max      40000.0
Name: price, dtype: float64

In [9]:
listings['price'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]).round(0)

0.10     691.0
0.25    1071.0
0.50    1791.0
0.75    3004.0
0.90    4446.0
0.99    8575.0
Name: price, dtype: float64

### IQR: interquartile range

In [10]:
q1 = listings['price'].quantile(0.25)
q3 = listings['price'].quantile(0.75)

iqr = q3 - q1

print(round(q1, 1), round(q3, 1), round(iqr, 1))


1071.1 3003.8 1932.7


|  | Typical value | Spread |
|---|---|---|
| Sensitive to extreme values | mean | standard deviation |
| Hard to push around | median | IQR |

## Outliers: what do we do with the 40,000

fences

In [11]:
# Calculate the two fences using IQR rule

upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr

print(round(lower, 1), round(upper, 1))

-1827.9 5902.8


In [12]:
outliers = listings[listings['price'] > upper]

print(len(outliers))
outliers['price'].describe().round(0)

132


count      132.0
mean      8817.0
std       5180.0
min       5910.0
25%       6305.0
50%       7091.0
75%       8354.0
max      40000.0
Name: price, dtype: float64

In [13]:
print(outliers['room_type'].value_counts())
print(outliers['accommodates'].median(), listings['accommodates'].median())
print((outliers['accommodates'] >= 6).mean().round(2), (listings[listings['price'] <= upper]['accommodates'] >= 6).mean().round(2))

room_type
Entire home/apt    128
Private room         4
Name: count, dtype: int64
6.0 3.0
0.68 0.17


In [14]:
listings['price_outlier'] = listings['price'] > upper

print(listings['price_outlier'].sum())

132


In [15]:
compare = pd.DataFrame({
    'all': listings['price'].agg(['count', 'mean', 'median', 'std']),
    'without_outliers' : listings.loc[~listings['price_outlier'], 'price'].agg(['count', 'mean', 'median', 'std']),
}).round(0)

compare

,all,without_outliers
count,2957.0,2825.0
mean,2328.0,2024.0
median,1791.0,1700.0
std,2160.0,1257.0


## Describing groups

In [16]:
listings.groupby('room_type')['price'].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
room_type,,,,,,,,
Entire home/apt,2346.0,2679.0,2208.0,264.0,1372.0,2168.0,3371.0,40000.0
Hotel room,1.0,859.0,NaN,859.0,859.0,859.0,859.0,859.0
Private room,591.0,999.0,1275.0,120.0,564.0,808.0,1129.0,25998.0
Shared room,19.0,406.0,293.0,253.0,288.0,302.0,320.0,1373.0


## Correlation: how two columns move together

Correlation lies between -1 and 1

- **Close to 1:** when one column goes up, the other goes up too. They move together.
- **Close to 0:** no connection. Knowing one column tells you nothing about the other.
- **Close to −1:** when one goes up, the other goes down.


corr()

In [17]:
columns = ['price', 'accommodates', 'bedrooms', 'beds', 'number_of_reviews', 'availability_365', 'review_scores_rating']

listings[columns].corr()['price'].round(2)

price                   1.00
accommodates            0.58
bedrooms                0.58
beds                    0.52
number_of_reviews      -0.09
availability_365       -0.00
review_scores_rating    0.17
Name: price, dtype: float64

In [18]:
listings.groupby('accommodates')['price'].median().head(8).round(0)

accommodates
1     673.0
2    1211.0
3    1460.0
4    2038.0
5    2997.0
6    3226.0
7    4027.0
8    4338.0
Name: price, dtype: float64

In [19]:
no_outliers = listings[~listings['price_outlier']]

print(listings['price'].corr(listings['accommodates']).round(2))
print(no_outliers['price'].corr(no_outliers['accommodates']).round(2))

0.58
0.65


In [20]:
print(listings['accommodates'].corr(listings['beds']).round(2))

0.82


## Correlation is not an explanation

correlation is not causation

In [21]:
# do superhosts and other hosts have the same *mix* of room types?

is_entire = listings['room_type'] == 'Entire home/apt'

(is_entire.groupby(listings['host_is_superhost']).mean() * 100).round(0)

host_is_superhost
False    84.0
True     74.0
Name: room_type, dtype: float64

In [22]:
listings[is_entire].groupby('host_is_superhost')['price'].agg(['count', 'median']).round(0)

,count,median
host_is_superhost,,
False,1887,2193.0
True,459,2111.0


cofunder

Q: What else could explain it